# 03 · Reliability and Bayesian updating

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
from offshore_risk import load_config
cfg = load_config()
from offshore_risk.reliability import models as rm
th = cfg.registry.at_quantile(1, 0.5)
shape, scale = float(th['comp_weibull_shape'][0]), float(th['comp_weibull_scale_years'][0])
ages = np.arange(0, 6)
pd.DataFrame({'age since overhaul (y)': ages, 'expected failures next year (Weibull NHPP)': rm.nhpp_expected_failures(ages, ages + 1, shape, scale),
              'constant-rate equivalent': 1 / float(rm.weibull_mean(shape, scale))})

,age since overhaul (y),expected failures next year (Weibull NHPP),constant-rate equivalent
0,0,0.118780,0.308047
1,1,0.254580,0.308047
2,2,0.356230,0.308047
3,3,0.443989,0.308047
4,4,0.523236,0.308047
5,5,0.596500,0.308047


With a wear-out shape (β > 1) the expected number of failures in the coming year depends on time since overhaul; a constant-rate model would give the same answer at every age and could not value maintenance timing.

In [2]:
lam, T = 2.5e-6, 8760
rows = [{'architecture': a, 'beta': b, 'PFDavg': float(rm.pfd_koon(k, n, lam, T, b))} for a, k, n in (('1oo1', 1, 1), ('1oo2', 1, 2), ('2oo3', 2, 3)) for b in (0.0, 0.05, 0.10)]
pd.DataFrame(rows).pivot(index='architecture', columns='beta', values='PFDavg')

beta,0.00,0.05,0.10
architecture,,,
1oo1,0.01095,0.010950,0.010950
1oo2,0.00016,0.000692,0.001224
2oo3,0.00048,0.000980,0.001483


Common cause dominates: for 1oo2 valves the β = 10 % term is several times the independent term, so adding redundancy without addressing common cause buys little.

In [3]:
lam_g, mttr, beta = 1.5e-4, 24, 0.05
pd.DataFrame([{'generators': f'{k}oo{n}', 'independent /yr': 8760 * rm.repairable_koon_failure_frequency(k, n, lam_g, mttr, 0),
              'with CCF /yr': 8760 * rm.repairable_koon_failure_frequency(k, n, lam_g, mttr, beta)} for k, n in ((2, 3), (2, 4))])

,generators,independent /yr,with CCF /yr
0,2oo3,0.028382,0.091315
1,2oo4,0.000204,0.065875


A fourth generator removes most of the *independent* loss-of-power frequency but leaves the common-cause term (shared fuel gas, switchboard) untouched, which is why that control scores poorly in the mitigation analysis.

## Bayesian updating from synthetic records

In [4]:
from offshore_risk.bayesian.calibration import calibrate, summary_table
cal = calibrate(cfg)
summary_table(cal)

,parameter,model,observed_failures,exposure,exposure_unit,mle,prior_mean,prior_90ci,posterior_mean,posterior_90ci,ci_width_reduction,fed_back_to_model
0,fw_elec_pump_fts,beta-binomial,36,1252.0,demands,0.028754,0.010000,"[0.000518, 0.0298]",0.027367,"[0.0205, 0.035]",0.503255,True
1,fw_diesel_pump_fts,beta-binomial,8,626.0,demands,0.012780,0.020000,"[0.00105, 0.0593]",0.013314,"[0.00697, 0.0213]",0.754380,True
2,spurious_trip_rate,gamma-poisson,58,12.0,complex-years,4.833333,3.278555,"[1.31, 5.96]",4.653491,"[3.73, 5.66]",0.586131,True
3,total_release_rate,gamma-poisson (display only),11,12.0,complex-years,0.916667,1.192814,"[0.505, 2.12]",0.995132,"[0.631, 1.43]",0.506366,False


In [5]:
from offshore_risk.bayesian.conjugate import BetaModel
from offshore_risk.visualization import plots
c = cal['fw_elec_pump_fts']; grid = np.linspace(1e-5, 0.06, 500)
plots.bayes_update(c['prior'], c['posterior'], lambda x: BetaModel.likelihood_density(x, c['failures'], c['exposure']), grid, 'Electric firewater pump fail-to-start', 'P(fail to start)')

<Figure size 825x418 with 1 Axes>

The generic prior (mean 1 %) was optimistic for these synthetic pumps; about 1,250 tests move the posterior to ~2.7 % and cut the 90 % interval width roughly in half. The posterior (not the prior) is what the main analysis uses.